# 04 VPD NASA POWER - AgroField AI

Objetivo: Calcular el VPD a partir de los datos climáticos originales (T2M y RH2M).

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
import matplotlib.pyplot as plt

## Definición de calculate_vpd()

In [ ]:
def calculate_vpd(T, RH):
    RH_decimal = RH / 100
    es = 0.6108 * np.exp((17.27 * T) / (T + 237.3))
    vpd = es * (1 - RH_decimal)
    return vpd

## Validación del cálculo individual

In [ ]:
T_test = 13.61
RH_test = 91.29
vpd_test = calculate_vpd(T_test, RH_test)
print(f'VPD calculado: {vpd_test:.4f} kPa') # Esperado ~0.1358

## Carga del JSON RAW

In [ ]:
with open('../data/raw/nasa_power/POWER_Point_Daily_20260724_20260822_001d72S_078d76W_LST.json', 'r') as f:
    data = json.load(f)
print('Claves principales:', data.keys())

## Conversión a DataFrame

In [ ]:
params = data['properties']['parameter']
df = pd.DataFrame(params)
df = df.reset_index().rename(columns={'index': 'fecha'})
df.head()

## Limpieza de valores faltantes

In [ ]:
df = df.replace(-999.0, np.nan)
df.isna().sum()

## Conversión de fechas

In [ ]:
df['fecha'] = pd.to_datetime(df['fecha'], format='%Y%m%d')
df.head()

## Cálculo de VPD

In [ ]:
df['VPD'] = calculate_vpd(df['T2M'], df['RH2M'])

## Validación del 17/08/2026

In [ ]:
row = df[df['fecha'] == '2026-08-17'].iloc[0]
print('T2M:', row['T2M'])
print('RH2M:', row['RH2M'])
print('VPD:', row['VPD'])

## Exploración de estadísticas y Guardado

In [ ]:
print(df.describe())

In [ ]:
df.to_csv('../data/processed/nasa_power_vpd_colta_20260724_20260822.csv', index=False)